# Self-Supervised Pretraining for Prostate Segmentation on Micro-Ultrasound Images
### Giorgi Karazanashvili | CS675 Computer Vision, UMass Boston

---
## 1. Setup and Data Loading

Download the Micro-Ultrasound Prostate Segmentation Dataset from Zenodo and inspect the volume structure.

In [ ]:
!pip install nibabel -q

import os
import zipfile
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from skimage.transform import resize
from scipy.ndimage import distance_transform_edt

# Download dataset
!wget -q "https://zenodo.org/records/10475293/files/Micro_Ultrasound_Prostate_Segmentation_Dataset.zip?download=1" -O dataset.zip
with zipfile.ZipFile("dataset.zip", "r") as z:
    z.extractall("data")
os.remove("dataset.zip")

BASE = "data/Micro_Ultrasound_Prostate_Segmentation_Dataset"
TRAIN_SCAN_DIR = f"{BASE}/train/micro_ultrasound_scans"
TRAIN_ANN_DIR = f"{BASE}/train/expert_annotations"
TEST_SCAN_DIR = f"{BASE}/test/micro_ultrasound_scans"
TEST_ANN_DIR = f"{BASE}/test/expert_annotations"

print(f"Train: {len(os.listdir(TRAIN_SCAN_DIR))} scans, {len(os.listdir(TRAIN_ANN_DIR))} annotations")
print(f"Test: {len(os.listdir(TEST_SCAN_DIR))} scans, {len(os.listdir(TEST_ANN_DIR))} annotations")

# Check volume shapes
for f in sorted(os.listdir(TRAIN_SCAN_DIR))[:3]:
    v = nib.load(os.path.join(TRAIN_SCAN_DIR, f))
    print(f"  {f}: {v.shape}")

---
## 2. Visualize a Sample Slice

Each volume is 1372 x 962 pixels with 33-45 axial slices. Expert annotations are binary masks.

In [ ]:
scan_vol = nib.load(os.path.join(TRAIN_SCAN_DIR, "microUS_train_01.nii.gz")).get_fdata()
ann_vol = nib.load(os.path.join(TRAIN_ANN_DIR, "expert_annotation_train_01.nii.gz")).get_fdata()

mid = scan_vol.shape[2] // 2

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(scan_vol[:, :, mid].T, cmap="gray", origin="lower")
axes[0].set_title("Scan")
axes[1].imshow(ann_vol[:, :, mid].T, cmap="gray", origin="lower")
axes[1].set_title("Expert annotation")
axes[2].imshow(scan_vol[:, :, mid].T, cmap="gray", origin="lower")
axes[2].imshow(ann_vol[:, :, mid].T, alpha=0.3, cmap="Reds", origin="lower")
axes[2].set_title("Overlay")
plt.tight_layout()
plt.show()

---
## 3. Dataset Class

Each input is a 3-channel image formed by stacking the previous, current, and next axial slice. All slices are resized to 256 x 256 and normalized to [0, 1]. Masks are resized with nearest-neighbor interpolation.

In [ ]:
class MicroUSDataset(Dataset):
    """2D slice dataset from micro-US volumes.
    Stacks adjacent slices as 3 channels for spatial context."""

    def __init__(self, scan_dir, ann_dir=None, target_size=(256, 256), file_list=None):
        self.target_size = target_size
        self.has_masks = ann_dir is not None
        self.slices = []

        scan_files = sorted(file_list) if file_list else sorted(os.listdir(scan_dir))
        for sf in scan_files:
            scan_p = os.path.join(scan_dir, sf)
            vol = nib.load(scan_p)
            n_slices = vol.shape[2]

            ann_p = None
            if self.has_masks:
                case_num = sf.split("_")[-1]
                ann_candidates = [f for f in os.listdir(ann_dir) if f.endswith(case_num)]
                if ann_candidates:
                    ann_p = os.path.join(ann_dir, ann_candidates[0])

            for i in range(n_slices):
                self.slices.append((scan_p, ann_p, i, n_slices))

        self._cache = {}

    def _load_volume(self, path):
        if path not in self._cache:
            self._cache[path] = nib.load(path).get_fdata()
        return self._cache[path]

    def __len__(self):
        return len(self.slices)

    def __getitem__(self, idx):
        scan_p, ann_p, si, n_slices = self.slices[idx]
        vol = self._load_volume(scan_p)

        prev_i = max(0, si - 1)
        next_i = min(n_slices - 1, si + 1)

        slices_3ch = np.stack([
            vol[:, :, prev_i].T,
            vol[:, :, si].T,
            vol[:, :, next_i].T
        ], axis=0)

        slices_3ch = slices_3ch / 255.0
        slices_3ch = np.transpose(slices_3ch, (1, 2, 0))
        slices_3ch = resize(slices_3ch, (*self.target_size, 3), anti_aliasing=True)
        slices_3ch = np.transpose(slices_3ch, (2, 0, 1))

        image = torch.from_numpy(slices_3ch).float()

        if self.has_masks:
            mask = self._load_volume(ann_p)[:, :, si].T
            mask = resize(mask, self.target_size, order=0, anti_aliasing=False)
            mask = torch.from_numpy(mask).float().unsqueeze(0)
            return image, mask

        return image

---
## 4. Patient-Level Split

All 55 training patients are used for SSL pretraining (no masks). 45 are used for supervised fine-tuning and 10 for validation. The 20 test patients are held out entirely.

In [ ]:
all_scans = sorted(os.listdir(TRAIN_SCAN_DIR))
np.random.seed(42)
perm = np.random.permutation(len(all_scans))
val_files = [all_scans[i] for i in perm[:10]]
train_files = [all_scans[i] for i in perm[10:]]

ssl_ds = MicroUSDataset(TRAIN_SCAN_DIR, ann_dir=None, file_list=all_scans)
ft_train_ds = MicroUSDataset(TRAIN_SCAN_DIR, TRAIN_ANN_DIR, file_list=train_files)
ft_val_ds = MicroUSDataset(TRAIN_SCAN_DIR, TRAIN_ANN_DIR, file_list=val_files)
test_ds = MicroUSDataset(TEST_SCAN_DIR, TEST_ANN_DIR)

print(f"SSL pretraining: {len(ssl_ds)} slices (all 55 patients, no masks)")
print(f"Fine-tune train: {len(ft_train_ds)} slices ({len(train_files)} patients)")
print(f"Fine-tune val:   {len(ft_val_ds)} slices ({len(val_files)} patients)")
print(f"Test:            {len(test_ds)} slices (20 patients)")

---
## 5. Model Architecture

The encoder has 4 downsampling stages (64/128/256/512 channels). For autoencoder pretraining, the decoder uses transposed convolutions without skip connections, forcing the bottleneck to encode all reconstruction information. For segmentation, a U-Net decoder with skip connections is used.

In [ ]:
class Encoder(nn.Module):
    """Convolutional encoder with 4 downsampling stages."""
    def __init__(self, in_channels=3):
        super().__init__()
        self.enc1 = self._block(in_channels, 64)
        self.enc2 = self._block(64, 128)
        self.enc3 = self._block(128, 256)
        self.enc4 = self._block(256, 512)
        self.pool = nn.MaxPool2d(2)

    def _block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        return e1, e2, e3, e4


class AEDecoder(nn.Module):
    """Decoder for autoencoder pretraining (no skip connections)."""
    def __init__(self, out_channels=3):
        super().__init__()
        self.up4 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = self._block(256, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = self._block(128, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = self._block(64, 64)
        self.final = nn.Conv2d(64, out_channels, 1)

    def _block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )

    def forward(self, e4):
        d = self.dec3(self.up4(e4))
        d = self.dec2(self.up3(d))
        d = self.dec1(self.up2(d))
        return torch.sigmoid(self.final(d))


class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder(in_channels=3)
        self.decoder = AEDecoder(out_channels=3)

    def forward(self, x):
        _, _, _, e4 = self.encoder(x)
        return self.decoder(e4)


class UNetDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.up4 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = self._block(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = self._block(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = self._block(128, 64)
        self.final = nn.Conv2d(64, 1, 1)

    def _block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )

    def forward(self, e1, e2, e3, e4):
        d = torch.cat([self.up4(e4), e3], dim=1)
        d = self.dec3(d)
        d = torch.cat([self.up3(d), e2], dim=1)
        d = self.dec2(d)
        d = torch.cat([self.up2(d), e1], dim=1)
        d = self.dec1(d)
        return self.final(d)


class UNet(nn.Module):
    def __init__(self, encoder, pretrained_path=None):
        super().__init__()
        self.encoder = encoder
        self.decoder = UNetDecoder()
        if pretrained_path:
            self.encoder.load_state_dict(torch.load(pretrained_path, weights_only=True))
            print("Loaded pretrained encoder weights.")

    def forward(self, x):
        e1, e2, e3, e4 = self.encoder(x)
        return self.decoder(e1, e2, e3, e4)


def dice_loss(pred, target, smooth=1.0):
    pred = torch.sigmoid(pred)
    intersection = (pred * target).sum(dim=(2, 3))
    union = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    dice = (2.0 * intersection + smooth) / (union + smooth)
    return 1.0 - dice.mean()


def hausdorff_95(pred, target):
    pred = pred.astype(bool)
    target = target.astype(bool)

    if not pred.any() and not target.any():
        return 0.0
    if not pred.any() or not target.any():
        return np.inf

    dt_pred = distance_transform_edt(~pred)
    dt_target = distance_transform_edt(~target)

    d_pred_to_target = dt_target[pred]
    d_target_to_pred = dt_pred[target]

    all_distances = np.concatenate([d_pred_to_target, d_target_to_pred])
    return np.percentile(all_distances, 95)

# Shape check
model = ConvAutoencoder()
dummy = torch.randn(2, 3, 256, 256)
out = model(dummy)
print(f"Autoencoder: {dummy.shape} -> {out.shape}")

unet = UNet(Encoder())
out = unet(dummy)
print(f"U-Net: {dummy.shape} -> {out.shape}")

---
## 6. Autoencoder Pretraining

Preprocess all SSL slices to disk for faster loading, then train the autoencoder for 50 epochs with MSE loss, Adam optimizer (lr=1e-3), and cosine annealing.

In [ ]:
PREPROCESSED_DIR = "preprocessed_slices"
os.makedirs(PREPROCESSED_DIR, exist_ok=True)

print("Preprocessing all SSL slices to disk...")
for i in range(len(ssl_ds)):
    save_path = os.path.join(PREPROCESSED_DIR, f"slice_{i:05d}.pt")
    if not os.path.exists(save_path):
        img = ssl_ds[i]
        torch.save(img, save_path)
    if (i + 1) % 500 == 0:
        print(f"  {i+1}/{len(ssl_ds)}")
print("Done.")

class PreprocessedDataset(Dataset):
    def __init__(self, directory):
        self.files = sorted([
            os.path.join(directory, f) for f in os.listdir(directory) if f.endswith(".pt")
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        return torch.load(self.files[idx])

ssl_ds_fast = PreprocessedDataset(PREPROCESSED_DIR)
print(f"Preprocessed slices: {len(ssl_ds_fast)}")

In [ ]:
def train_autoencoder(model, train_ds, num_epochs=50, batch_size=16, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    history = []
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        for batch in loader:
            batch = batch.to(device)
            recon = model(batch)
            loss = nn.functional.mse_loss(recon, batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * batch.size(0)

        epoch_loss /= len(train_ds)
        scheduler.step()
        history.append(epoch_loss)

        print(f"Epoch {epoch+1}/{num_epochs} -- MSE: {epoch_loss:.6f}")
    return history

ae_model = ConvAutoencoder()
ae_history = train_autoencoder(ae_model, ssl_ds_fast, num_epochs=50, batch_size=16, lr=1e-3)

---
## 7. Pretraining Results

Plot the loss curve and visualize reconstruction quality.

In [ ]:
# Loss curve
plt.figure(figsize=(8, 4))
plt.plot(ae_history)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Autoencoder Pretraining Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Reconstructions
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ae_model.eval()

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
indices = np.linspace(0, len(ssl_ds_fast)-1, 5, dtype=int)

with torch.no_grad():
    for i, idx in enumerate(indices):
        img = ssl_ds_fast[idx].unsqueeze(0).to(device)
        recon = ae_model(img)

        axes[0, i].imshow(img[0, 1].cpu(), cmap="gray")
        axes[0, i].set_title("Original")
        axes[0, i].axis("off")

        axes[1, i].imshow(recon[0, 1].cpu(), cmap="gray")
        axes[1, i].set_title("Reconstruction")
        axes[1, i].axis("off")

plt.suptitle("Autoencoder Reconstructions")
plt.tight_layout()
plt.show()

# Save encoder weights
torch.save(ae_model.encoder.state_dict(), "pretrained_encoder.pth")
print("Encoder weights saved.")

---
## 8. U-Net Training

Train both the pretrained and scratch U-Net on the full 45-patient labeled set for 100 epochs. Loss is Dice + BCE, Adam optimizer with lr=1e-4, cosine annealing schedule.

In [ ]:
def train_unet(model, train_ds, val_ds, num_epochs=100, batch_size=16, lr=1e-4, save_prefix="model"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    best_val_dice = 0
    history = {"train_loss": [], "val_dice": []}

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)
            loss = nn.functional.binary_cross_entropy_with_logits(logits, masks) + dice_loss(logits, masks)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * imgs.size(0)

        epoch_loss /= len(train_ds)
        scheduler.step()

        model.eval()
        val_dice_sum = 0
        val_count = 0
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                preds = (torch.sigmoid(model(imgs)) > 0.5).float()
                intersection = (preds * masks).sum(dim=(2, 3))
                union = preds.sum(dim=(2, 3)) + masks.sum(dim=(2, 3))
                dice = (2.0 * intersection + 1.0) / (union + 1.0)
                val_dice_sum += dice.sum().item()
                val_count += imgs.size(0)

        val_dice = val_dice_sum / val_count
        history["train_loss"].append(epoch_loss)
        history["val_dice"].append(val_dice)

        if val_dice > best_val_dice:
            best_val_dice = val_dice
            torch.save(model.state_dict(), f"best_{save_prefix}.pth")

        print(f"Epoch {epoch+1}/{num_epochs} -- Loss: {epoch_loss:.4f}, Val Dice: {val_dice:.4f}")

    print(f"Best Val Dice: {best_val_dice:.4f}")
    return history

In [ ]:
# Pretrained U-Net
pretrained_unet = UNet(Encoder(), pretrained_path="pretrained_encoder.pth")
pretrained_history = train_unet(pretrained_unet, ft_train_ds, ft_val_ds, save_prefix="unet_pretrained")

In [ ]:
# Scratch U-Net
scratch_unet = UNet(Encoder(), pretrained_path=None)
scratch_history = train_unet(scratch_unet, ft_train_ds, ft_val_ds, save_prefix="unet_scratch")

---
## 9. Test Evaluation

Evaluate both models on the held-out 20-patient test set using Dice and HD95.

In [ ]:
def evaluate_on_test(model, test_ds, device):
    model.eval()
    loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=0)

    dice_scores = []
    hd95_scores = []

    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(device)
            preds = (torch.sigmoid(model(imgs)) > 0.5).float().cpu().numpy()
            masks = masks.numpy()

            for i in range(preds.shape[0]):
                p = preds[i, 0]
                m = masks[i, 0]

                intersection = (p * m).sum()
                union = p.sum() + m.sum()
                dice = (2.0 * intersection + 1.0) / (union + 1.0)
                dice_scores.append(dice)

                hd = hausdorff_95(p, m)
                if np.isfinite(hd):
                    hd95_scores.append(hd)

    return {
        "dice_mean": np.mean(dice_scores),
        "dice_std": np.std(dice_scores),
        "hd95_mean": np.mean(hd95_scores),
        "hd95_std": np.std(hd95_scores)
    }

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pretrained_unet.load_state_dict(torch.load("best_unet_pretrained.pth", weights_only=True))
scratch_unet.load_state_dict(torch.load("best_unet_scratch.pth", weights_only=True))

pretrained_results = evaluate_on_test(pretrained_unet, test_ds, device)
scratch_results = evaluate_on_test(scratch_unet, test_ds, device)

print("=== Test Set Results (45 patients) ===")
print(f"Pretrained -- Dice: {pretrained_results['dice_mean']:.4f} +/- {pretrained_results['dice_std']:.4f}, HD95: {pretrained_results['hd95_mean']:.2f} +/- {pretrained_results['hd95_std']:.2f}")
print(f"Scratch    -- Dice: {scratch_results['dice_mean']:.4f} +/- {scratch_results['dice_std']:.4f}, HD95: {scratch_results['hd95_mean']:.2f} +/- {scratch_results['hd95_std']:.2f}")

---
## 10. Label-Efficiency Experiment

Train both models at 5, 10, and 20 labeled patients and compare against the full 45-patient results.

In [ ]:
label_counts = [5, 10, 20]
results = {
    "pretrained": {"45": pretrained_results},
    "scratch": {"45": scratch_results}
}
all_histories = {}

for n_patients in label_counts:
    print(f"\n{'='*50}")
    print(f"Training with {n_patients} labeled patients")
    print(f"{'='*50}")

    subset_files = train_files[:n_patients]
    subset_ds = MicroUSDataset(TRAIN_SCAN_DIR, TRAIN_ANN_DIR, file_list=subset_files)
    print(f"Training slices: {len(subset_ds)}")

    for model_type in ["pretrained", "scratch"]:
        print(f"\n--- {model_type} ---")

        if model_type == "pretrained":
            model = UNet(Encoder(), pretrained_path="pretrained_encoder.pth")
        else:
            model = UNet(Encoder(), pretrained_path=None)

        h = train_unet(model, subset_ds, ft_val_ds, save_prefix=f"{model_type}_{n_patients}p")
        all_histories[f"{model_type}_{n_patients}p"] = h

        model.load_state_dict(torch.load(f"best_{model_type}_{n_patients}p.pth", weights_only=True))
        result = evaluate_on_test(model, test_ds, device)
        results[model_type][str(n_patients)] = result
        print(f"Test Dice: {result['dice_mean']:.4f}, HD95: {result['hd95_mean']:.2f}")

# Summary
print(f"\n{'='*60}")
print("LABEL EFFICIENCY SUMMARY")
print(f"{'='*60}")
print(f"{'Patients':<12}{'Pretrained Dice':<20}{'Scratch Dice':<20}{'Difference'}")
for n in ["5", "10", "20", "45"]:
    p = results["pretrained"][n]["dice_mean"]
    s = results["scratch"][n]["dice_mean"]
    print(f"{n:<12}{p:.4f}{'':<14}{s:.4f}{'':<14}{p-s:+.4f}")

print(f"\n{'Patients':<12}{'Pretrained HD95':<20}{'Scratch HD95':<20}{'Difference'}")
for n in ["5", "10", "20", "45"]:
    p = results["pretrained"][n]["hd95_mean"]
    s = results["scratch"][n]["hd95_mean"]
    print(f"{n:<12}{p:.2f}{'':<14}{s:.2f}{'':<14}{p-s:+.2f}")

---
## 11. Results Visualization

In [ ]:
# Label-efficiency curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

patients = [5, 10, 20, 45]
pre_dice = [results["pretrained"][str(n)]["dice_mean"] for n in patients]
scr_dice = [results["scratch"][str(n)]["dice_mean"] for n in patients]
pre_hd = [results["pretrained"][str(n)]["hd95_mean"] for n in patients]
scr_hd = [results["scratch"][str(n)]["hd95_mean"] for n in patients]

ax1.plot(patients, pre_dice, "o-", label="Pretrained")
ax1.plot(patients, scr_dice, "s--", label="Scratch")
ax1.set_xlabel("Number of Labeled Patients")
ax1.set_ylabel("Test Dice Score")
ax1.set_title("Label Efficiency -- Dice")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xticks(patients)

ax2.plot(patients, pre_hd, "o-", label="Pretrained")
ax2.plot(patients, scr_hd, "s--", label="Scratch")
ax2.set_xlabel("Number of Labeled Patients")
ax2.set_ylabel("HD95 (pixels)")
ax2.set_title("Label Efficiency -- HD95")
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xticks(patients)

plt.tight_layout()
plt.show()

In [ ]:
# Training curves at reduced label counts
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, n in enumerate([5, 10, 20]):
    for model_type, style in [("pretrained", "-"), ("scratch", "--")]:
        key = f"{model_type}_{n}p"
        h = all_histories[key]
        axes[0, i].plot(h["train_loss"], style, label=model_type)
        axes[1, i].plot(h["val_dice"], style, label=model_type)

    axes[0, i].set_title(f"{n} patients -- Loss")
    axes[0, i].set_xlabel("Epoch")
    axes[0, i].legend()
    axes[0, i].grid(True, alpha=0.3)

    axes[1, i].set_title(f"{n} patients -- Val Dice")
    axes[1, i].set_xlabel("Epoch")
    axes[1, i].legend()
    axes[1, i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Qualitative segmentation results
fig, axes = plt.subplots(4, 3, figsize=(12, 16))

pretrained_unet.eval()
scratch_unet.eval()

sample_indices = np.linspace(100, 600, 4, dtype=int)

for row, idx in enumerate(sample_indices):
    img, mask = test_ds[idx]
    img_dev = img.unsqueeze(0).to(device)

    with torch.no_grad():
        pre_pred = (torch.sigmoid(pretrained_unet(img_dev)) > 0.5).float().cpu()[0, 0]
        scr_pred = (torch.sigmoid(scratch_unet(img_dev)) > 0.5).float().cpu()[0, 0]

    axes[row, 0].imshow(img[1], cmap="gray")
    axes[row, 0].imshow(mask[0], alpha=0.3, cmap="Reds")
    axes[row, 0].set_title("Ground Truth")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(img[1], cmap="gray")
    axes[row, 1].imshow(pre_pred, alpha=0.3, cmap="Reds")
    axes[row, 1].set_title("Pretrained")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(img[1], cmap="gray")
    axes[row, 2].imshow(scr_pred, alpha=0.3, cmap="Reds")
    axes[row, 2].set_title("Scratch")
    axes[row, 2].axis("off")

plt.suptitle("Test Set Segmentation Examples (45 patients)")
plt.tight_layout()
plt.show()